# 02 — Exploratory Data Analysis
**Telco Customer Churn Analysis · Future Interns Task 2 · 2026**

---

## Objective
Explore churn patterns across every major dimension of the dataset. Each chart is followed by a **business interpretation** — this is what separates an analyst from someone who just plots data.

> *"If I were the product manager, what would I want to know?"*

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Style
plt.rcParams.update({
    'figure.facecolor': '#f8f9fa',
    'axes.facecolor': '#f8f9fa',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': '--'
})

COLORS = {
    'churn':    '#d13438',
    'retain':   '#13a10e',
    'neutral':  '#118dff',
    'warning':  '#f2c811',
    'palette':  ['#d13438', '#f2c811', '#13a10e', '#118dff', '#744da9', '#12b2d0']
}

df = pd.read_csv('../data/telco_churn_clean.csv')
print(f"Dataset loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Overall churn rate: {df['Churn_bin'].mean()*100:.1f}%")

FileNotFoundError: [Errno 2] No such file or directory: '../data/telco_churn_clean.csv'

## 1. Churn by Contract Type

The first and most important question: does the type of commitment a customer makes predict whether they stay?

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: churn rate by contract
contract_churn = df.groupby('Contract')['Churn_bin'].mean().sort_values(ascending=False) * 100
bars = axes[0].bar(contract_churn.index, contract_churn.values,
                   color=[COLORS['churn'], COLORS['warning'], COLORS['retain']],
                   edgecolor='none', width=0.55)
axes[0].set_title('Churn Rate by Contract Type', fontweight='bold', fontsize=12)
axes[0].set_ylabel('Churn Rate (%)')
axes[0].set_ylim(0, 55)
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                 f'{bar.get_height():.1f}%', ha='center', fontweight='bold', fontsize=11)

# Right: customer count by contract
contract_counts = df.groupby('Contract').size()
axes[1].bar(contract_counts.index, contract_counts.values,
            color=COLORS['palette'][:3], edgecolor='none', width=0.55)
axes[1].set_title('Customer Count by Contract Type', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Number of Customers')
for bar in axes[1].patches:
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
                 f'{int(bar.get_height()):,}', ha='center', fontsize=11)

plt.suptitle('Contract Type Analysis', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../reports/figures/01_contract_churn.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Business interpretation
print("=" * 60)
print("BUSINESS INSIGHT — Contract Type")
print("=" * 60)
for c, row in df.groupby('Contract')['Churn_bin'].agg(['mean','count']).iterrows():
    print(f"  {c:<20} Churn: {row['mean']*100:>5.1f}%   Customers: {int(row['count']):>5,}")

print()
m2m_rate = df[df['Contract']=='Month-to-month']['Churn_bin'].mean()*100
two_yr   = df[df['Contract']=='Two year']['Churn_bin'].mean()*100
print(f"Month-to-month is {m2m_rate/two_yr:.1f}x riskier than two-year contracts.")
print("Action: Incentivise contract upgrades at 60-90 days into service.")

## 2. Churn by Internet Service

Does the type of internet service affect retention?

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))

internet_churn = df.groupby('InternetService')['Churn_bin'].mean().sort_values(ascending=False) * 100
bars = ax.barh(internet_churn.index, internet_churn.values,
               color=[COLORS['churn'], COLORS['warning'], COLORS['retain']],
               edgecolor='none', height=0.5)
ax.set_xlabel('Churn Rate (%)')
ax.set_title('Churn Rate by Internet Service Type', fontweight='bold', fontsize=12)
ax.set_xlim(0, 55)
for bar in bars:
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{bar.get_width():.1f}%', va='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_internet_churn.png', dpi=150, bbox_inches='tight')
plt.show()

print("Fiber optic vs DSL churn ratio:",
      round(df[df['InternetService']=='Fiber optic']['Churn_bin'].mean() /
            df[df['InternetService']=='DSL']['Churn_bin'].mean(), 2), "x")

## 3. Monthly Charges Distribution: Churned vs Retained

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# KDE plot
for label, grp in df.groupby('Churn'):
    color = COLORS['churn'] if label == 'Yes' else COLORS['retain']
    axes[0].hist(grp['MonthlyCharges'], bins=30, alpha=0.6, color=color,
                 label=f"{'Churned' if label=='Yes' else 'Retained'}", density=True)
axes[0].set_title('Monthly Charges Distribution', fontweight='bold', fontsize=12)
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].set_ylabel('Density')
axes[0].legend()

# Box plot
df_box = df[['MonthlyCharges','Churn']].copy()
df_box['Churn_label'] = df_box['Churn'].map({'Yes':'Churned','No':'Retained'})
bp = axes[1].boxplot([df[df['Churn']=='No']['MonthlyCharges'],
                      df[df['Churn']=='Yes']['MonthlyCharges']],
                     labels=['Retained','Churned'],
                     patch_artist=True,
                     boxprops=dict(facecolor='#e8f4f8'))
axes[1].set_title('Monthly Charges: Box Plot', fontweight='bold', fontsize=12)
axes[1].set_ylabel('Monthly Charges ($)')

plt.tight_layout()
plt.savefig('../reports/figures/03_charges_dist.png', dpi=150, bbox_inches='tight')
plt.show()

print("Avg Monthly Charges:")
print(df.groupby('Churn')['MonthlyCharges'].mean().round(2))

## 4. Churn by Payment Method

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
pay_churn = df.groupby('PaymentMethod')['Churn_bin'].mean().sort_values(ascending=False) * 100
short_labels = [p.replace(' (automatic)', '\n(auto)') for p in pay_churn.index]
colors = [COLORS['churn'] if v > 30 else COLORS['warning'] if v > 20 else COLORS['retain']
          for v in pay_churn.values]
bars = ax.bar(short_labels, pay_churn.values, color=colors, edgecolor='none', width=0.55)
ax.set_title('Churn Rate by Payment Method', fontweight='bold', fontsize=12)
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)
ax.axhline(26.54, color='grey', linestyle='--', linewidth=1, label='Overall avg (26.5%)')
ax.legend()
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{bar.get_height():.1f}%', ha='center', fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('../reports/figures/04_payment_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Add-on Services Impact

In [ ]:
addons = ['OnlineSecurity', 'TechSupport', 'OnlineBackup', 'DeviceProtection']
fig, axes = plt.subplots(1, 4, figsize=(14, 4))

for i, col in enumerate(addons):
    rates = df[df[col].isin(['Yes','No'])].groupby(col)['Churn_bin'].mean() * 100
    colors = [COLORS['retain'] if idx=='Yes' else COLORS['churn'] for idx in rates.index]
    axes[i].bar(rates.index, rates.values, color=colors, edgecolor='none', width=0.5)
    axes[i].set_title(col, fontweight='bold', fontsize=10)
    axes[i].set_ylim(0, 55)
    axes[i].set_ylabel('Churn Rate (%)' if i == 0 else '')
    for bar in axes[i].patches:
        axes[i].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
                     f'{bar.get_height():.1f}%', ha='center', fontsize=10, fontweight='bold')

plt.suptitle('Impact of Add-on Services on Churn Rate', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/05_addons_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Senior Citizen Churn Comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
senior_churn = df.groupby('SeniorCitizen')['Churn_bin'].mean() * 100
labels = ['Non-Senior', 'Senior Citizen']
bars = ax.bar(labels, senior_churn.values,
              color=[COLORS['neutral'], COLORS['churn']],
              edgecolor='none', width=0.45)
ax.set_title('Churn Rate: Senior vs Non-Senior Citizens', fontweight='bold', fontsize=12)
ax.set_ylabel('Churn Rate (%)')
ax.set_ylim(0, 55)
ax.axhline(26.54, color='grey', linestyle='--', linewidth=1, label='Overall avg')
ax.legend()
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.8,
            f'{bar.get_height():.1f}%', ha='center', fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/06_senior_churn.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Churn Rate Summary — All Segments

In [ ]:
summary = {
    'Electronic check (payment)':    df[df['PaymentMethod']=='Electronic check']['Churn_bin'].mean()*100,
    'Month-to-month (contract)':     df[df['Contract']=='Month-to-month']['Churn_bin'].mean()*100,
    'Fiber optic (internet)':        df[df['InternetService']=='Fiber optic']['Churn_bin'].mean()*100,
    'Senior citizen':                df[df['SeniorCitizen']==1]['Churn_bin'].mean()*100,
    'No OnlineSecurity':             df[df['OnlineSecurity']=='No']['Churn_bin'].mean()*100,
    'No TechSupport':                df[df['TechSupport']=='No']['Churn_bin'].mean()*100,
    'New customer (0-12m)':          df[df['tenure_group']=='0-12m']['Churn_bin'].mean()*100,
    'Overall average':               df['Churn_bin'].mean()*100,
    'DSL (internet)':                df[df['InternetService']=='DSL']['Churn_bin'].mean()*100,
    'One year (contract)':           df[df['Contract']=='One year']['Churn_bin'].mean()*100,
    'Has OnlineSecurity':            df[df['OnlineSecurity']=='Yes']['Churn_bin'].mean()*100,
    'Has TechSupport':               df[df['TechSupport']=='Yes']['Churn_bin'].mean()*100,
    'Two year (contract)':           df[df['Contract']=='Two year']['Churn_bin'].mean()*100,
}
summary_s = pd.Series(summary).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors = [COLORS['churn'] if v > 30 else COLORS['warning'] if v > 20 else COLORS['retain']
          for v in summary_s.values]
ax.barh(summary_s.index, summary_s.values, color=colors, edgecolor='none', height=0.6)
ax.axvline(26.54, color='grey', linestyle='--', linewidth=1.2, label='Overall avg (26.5%)')
ax.set_xlabel('Churn Rate (%)')
ax.set_title('Churn Rate by Segment — Full Summary', fontweight='bold', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('../reports/figures/07_churn_summary.png', dpi=150, bbox_inches='tight')
plt.show()

## Key EDA Takeaways

| Finding | Churn Rate | vs Average |
|---|---|---|
| Electronic check payment | 45.3% | +71% |
| New customers (0–12m) | 47.7% | +80% |
| Month-to-month contract | 42.7% | +61% |
| Fiber optic internet | 41.9% | +58% |
| No OnlineSecurity | 41.8% | +58% |
| **Two-year contract** | **2.8%** | **−89%** |
| **Has TechSupport** | **15.2%** | **−43%** |

**Next:** `03_cohort_retention_analysis.ipynb` — cohort heatmap and lifetime value analysis.